In [1]:
# Copyright 2023 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Text Summarization with Generative Models on Vertex AI

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/language/examples/prompt-design/text_summarization.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/colab-logo-32px.png" alt="Google Colaboratory logo"><br> Run in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/language/examples/prompt-design/text_summarization.ipynb">
      <img src="https://cloud.google.com/ml-engine/images/github-logo-32px.png" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/blob/main/language/examples/prompt-design/text_summarization.ipynb">
      <img src="https://lh3.googleusercontent.com/UiNooY4LUgW_oTvpsNhPpQzsstV5W8F7rYgxgGBD85cWJoLmrOzhVs_ksK_vgx40SHs7jCqkTkCk=e14-rj-sc0xffffff-h130-w32" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
</table>


## Overview
Text summarization produces a concise and fluent summary of a longer text document. There are two main text summarization types: extractive and abstractive. Extractive summarization involves selecting critical sentences from the original text and combining them to form a summary. Abstractive summarization involves generating new sentences representing the original text's main points. In this notebook, you go through a few examples of how large language models can help with generating summaries based on text more specifically generating concise summaries of house listing.

Learn more about text summarization in the [official documentation](https://cloud.google.com/vertex-ai/docs/generative-ai/text/summarization-prompts).

### Objective

In this tutorial, you will learn how to use generative models to summarize information from text by working through the following examples:
- Transcript summarization
- Summarizing text into bullet points
- Hashtag tokenization
- Title & heading generation

You also learn how to evaluate model-generated summaries by comparing to human-created summaries using ROUGE as an evaluation framework.

### Costs

This tutorial uses billable components of Google Cloud:

* Vertex AI Generative AI Studio

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing),
and use the [Pricing Calculator](https://cloud.google.com/products/calculator/)
to generate a cost estimate based on your projected usage.

## Getting Started

### Install Vertex AI SDK

In [2]:
!pip install google-cloud-aiplatform --upgrade --user

**Colab only:** Uncomment the following cell to restart the kernel or use the button to restart the kernel. For Vertex AI Workbench you can restart the terminal using the button on top. 

In [ ]:
# # Automatically restart kernel after installs so that your environment can access the new packages
# import IPython

# app = IPython.Application.instance()
# app.kernel.do_shutdown(True)

### Authenticating your notebook environment
* If you are using **Colab** to run this notebook, uncomment the cell below and continue.
* If you are using **Vertex AI Workbench**, check out the setup instructions [here](#TODO).
* If you are using **local Jupyter**, check out the setup instructions [here](#TODO).

In [ ]:
# from google.colab import auth
# auth.authenticate_user()

### Import libraries

Let's start by importing the libraries that we will need for this tutorial

**Colab only:** Uncomment the following cell to initialize the Vertex AI SDK. For Vertex AI Workbench, you don't need to run this.  

In [ ]:
# import vertexai

# PROJECT_ID = "[your-project-id]"  # @param {type:"string"}
# vertexai.init(project=PROJECT_ID, location="us-central1")

In [1]:
from vertexai.preview.language_models import TextGenerationModel

### Import models

Here we load the pre-trained text generation model called `text-bison@001`.

In [2]:
generation_model = TextGenerationModel.from_pretrained("text-bison@001")

## Text Summarization

### Transcript summarization

In this first example, you summarize a piece of text on house listing.

In [3]:
prompt = """
Provide a very short summary, no more than three sentences, for the following text:

Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!

Summary:

"""

print(
    generation_model.predict(
        prompt, temperature=0.2, max_output_tokens=1024, top_k=40, top_p=0.8
    ).text
)

1234 Seaside Drive is a 2-story home with breathtaking ocean views from both levels. It features an open floor plan, vaulted beamed ceilings, fireplace, quartz island, top-of-the-line appliances, primary suite with ocean vistas, 2 bedrooms, stylish bathroom, expansive patio, and succulent garden.


Instead of a summary, we can ask for a TL;DR ("too long; didn't read"). You can compare the differences between the outputs generated.

In [4]:
prompt = """
Provide a TL;DR for the following text:

Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!


TL;DR:
"""

print(
    generation_model.predict(
        prompt, temperature=0.2, max_output_tokens=1024, top_k=40, top_p=0.8
    ).text
)

1234 Seaside Drive is a 2-story home with breathtaking ocean views from both levels. It has an open floor plan, vaulted beamed ceilings, a fireplace, a quartz island, top-of-the-line appliances, a primary suite with ocean vistas, vaulted ceilings, an en-suite bath with dual sinks and a glass-tiled walk-in shower, two bedrooms with ocean views, and a stylish bathroom. It is located in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks.


### Summarize text into bullet points
In the following example, you use same text on house listing, but ask the model to summarize it in bullet-point form. Feel free to change the prompt.

In [5]:
prompt = """
Provide a very short summary in four bullet points for the following text:

Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!

Bulletpoints:

"""

print(
    generation_model.predict(
        prompt, temperature=0.2, max_output_tokens=256, top_k=1, top_p=0.8
    ).text
)

- 2-story home with breathtaking ocean views from both levels
- Open floor plan with vaulted beamed ceilings and a charming fireplace
- Luxurious primary suite with ocean vistas, vaulted ceilings, and an en-suite bath
- Expansive patio, perfect for entertaining amidst a tasteful succulent garden


###  Hashtag tokenization
Hashtag tokenization is the process of taking a piece of text and getting the hashtag "tokens" out of it. You can use this, for example, if you want to generate hashtags for your social media campaigns. In this example, you take [this tweet from Google Cloud](https://twitter.com/googlecloud/status/1649127992348606469) and generate some hashtags you can use.

In [6]:
prompt = """
Tokenize the hashtags of this text:

Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!
"""

print(
    generation_model.predict(
        prompt, temperature=0.8, max_output_tokens=1024, top_k=40, top_p=0.8
    ).text
)

#dream #coastal #retreat


### Title & heading generation
Below, you ask the model to generate five options for possible title/heading combos for a given piece of text.

In [7]:
prompt = """
Write a title for this text, give me five options:

Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!
"""

print(
    generation_model.predict(
        prompt, temperature=0.8, max_output_tokens=256, top_k=1, top_p=0.8
    ).text
)

1. **1234 Seaside Drive: Breathtaking Ocean Views**
2. **Luxurious 2-Story Home with Ocean Views**
3. **Stunning Remodeled Home in Coastal Heights**
4. **Your Dream Coastal Retreat**
5. **Situated in the Heart of Coastal Heights**


## Evaluation
You can evaluate the outputs from summarization tasks using [ROUGE](https://en.wikipedia.org/wiki/ROUGE_(metric)) as an evalulation framework. ROUGE (Recall-Oriented Understudy for Gisting Evaluation) are measures to automatically determine the quality of a summary by comparing it to other (ideal) summaries created by humans. The measures count the number of overlapping units such as n-gram, word sequences, and word pairs between the computer-generated summary to be evaluated and the ideal summaries created by humans.


The first step is to install the ROUGE library.

In [8]:
!pip install rouge

Create a summary from a language model that you can use to compare against a human-generated summary.

In [9]:
from rouge import Rouge

ROUGE = Rouge()

prompt = """
Provide a very short, maximum four sentences, summary for the following article:

Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!

Summary:

"""

candidate = generation_model.predict(
    prompt, temperature=0.1, max_output_tokens=1024, top_k=40, top_p=0.9
).text

print(candidate)

1234 Seaside Drive is a 2-story home with breathtaking ocean views from both levels. It has an open floor plan, vaulted beamed ceilings, a fireplace, a quartz island, top-of-the-line appliances, a luxurious primary suite, two bedrooms, a stylish bathroom, an expansive patio, and a succulent garden.


You will also need a human-generated summary that we will use to compare to the `candidate` generated by the model. We will call this `reference`. 

In [ ]:
reference = "Introducing 1234 Seaside Drive, a beautifully remodeled 2-story home with breathtaking ocean views from both levels. The upper level boasts an inviting open floor plan with vaulted beamed ceilings and a charming fireplace. The kitchen features a stunning oversized quartz island and top-of-the-line appliances. The luxurious primary suite offers ocean vistas, vaulted ceilings, and an en-suite bath with dual sinks and a glass-tiled walk-in shower. Downstairs, two generously sized bedrooms with ocean views and a stylish bathroom await. Step outside to the expansive patio, perfect for entertaining amidst a tasteful succulent garden. Situated in the sought-after Coastal Heights neighborhood, just minutes from shops, dining, beaches, and natural parks. Your dream coastal retreat awaits!"

Now you can take the candidate and reference to evaluate the performance. In this case, ROUGE will give you:

- `rouge-1`, which measures unigram overlap
- `rouge-2`, which measures bigram overlap
- `rouge-l`, which measures the longest common subsequence

In [ ]:
ROUGE.get_scores(candidate, reference)